# Trabajo Práctico 2 - Grupo 02

### Modelo XGBoost

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen


El objetivo es entrenar un modelo de XGBoost sobre dos representaciones de texto (TF-IDF y BoW) con búsqueda de hiperparámetros, comparando su desempeño contra el baseline de Naive Bayes (~ 0.66 F1-macro en Kaggle) y contra el de Random Forest (~ 0.65 F1-macro en Kaggle).

XGBoost (eXtreme Gradient Boosting) es una técnica de aprendizaje por ensambles (ensemble learning), específicamente de tipo boosting. Combina múltiples modelos predictivos básicos (conocidos como "aprendices débiles"), generalmente árboles de decisión, para crear un modelo final altamente preciso y robusto.

## Importación e instalación de dependencias

In [1]:
!pip install click
!pip install -q spacy
!python -m spacy download es_core_news_sm
!pip install numpy pandas matplotlib seaborn xgboost scikit-learn
!pip install nltk

  Using cached es_core_news_sm-3.8.0-py3-none-any.whl (12.9 MB)
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight

In [3]:
import nltk
nltk.download("sentiwordnet")
nltk.download("wordnet")
nltk.download("omw-1.4")
from scipy.sparse import hstack, csr_matrix

[nltk_data] Downloading package sentiwordnet to
[nltk_data]     C:\Users\gonza\AppData\Roaming\nltk_data...
[nltk_data]   Package sentiwordnet is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\gonza\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\gonza\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [4]:
import sys
sys.path.insert(0, "../../../..")  # sube cuatro niveles: v8 → XGBoost → Entregas → TP2

In [5]:
from common.preprocessing import clean_classical
from common.data_utils import get_split, make_tfidf, make_bow, SEED, extract_features, VectorizerPlusFeaturesPipeline
from common.evaluation import evaluate
from common.io_utils import save_model, load_model, make_submission

np.random.seed(SEED)

## Carga de datos y preprocesamiento

In [6]:
train = pd.read_csv("../../../../data/train.csv")
test  = pd.read_csv("../../../../data/test.csv")
X_train_raw, X_val_raw, y_train, y_val = get_split(train)

In [7]:
print("Preprocesando")
X_train = np.array([clean_classical(t) for t in X_train_raw])
X_val   = np.array([clean_classical(t) for t in X_val_raw])
X_test  = np.array([clean_classical(t) for t in test['text'].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

Preprocesando
Train: 40,800 | Val: 10,200 | Test: 8,500


## Entrega N8.2: XGBoost + BoW con feature engineering pero sin optimización de hiperparámetros

In [8]:
pipe = VectorizerPlusFeaturesPipeline(
    vectorizer=make_bow(),
    classifier=XGBClassifier(
        max_depth=7,
        learning_rate=0.2,
        colsample_bytree=1.0,
        min_child_weight=1,
        random_state=SEED, n_jobs=-1, eval_metric="mlogloss"
    )
)

# Fijados por experimentos anteriores, solo exploramos lo que falta
param_dist = {
    "classifier__n_estimators": [400, 500, 600, 700],
    "classifier__subsample":    [0.65, 0.7, 0.75, 0.8],
    "classifier__reg_alpha":    [0, 0.01, 0.1],
    "classifier__reg_lambda":   [1, 1.5, 2],
}

weights_train = compute_sample_weight("balanced", y_train)

rs = RandomizedSearchCV(pipe, param_dist, n_iter=40, cv=5, scoring="f1_macro",
                        n_jobs=2, random_state=SEED, verbose=1)

X_train_combined = np.column_stack([X_train, X_train_raw])
X_val_combined   = np.column_stack([X_val, X_val_raw])
X_full_combined  = np.column_stack([np.concatenate([X_train, X_val]),
                                    np.concatenate([X_train_raw, X_val_raw])])

In [10]:
print("Buscando mejores hiperparámetros XGBoost + BoW + features")
rs.fit(X_train_combined, y_train, sample_weight=weights_train)

print(f"\nMejores hiperparámetros: {rs.best_params_}")
print(f"Mejor F1-macro en CV: {rs.best_score_:.4f}")

y_pred = rs.best_estimator_.predict(X_val_combined)
evaluate("xgb_bow_features_rs_v8_2", y_val, y_pred, hyperparams=rs.best_params_)

Buscando mejores hiperparámetros XGBoost + BoW + features
Fitting 5 folds for each of 40 candidates, totalling 200 fits

Mejores hiperparámetros: {'classifier__subsample': 0.75, 'classifier__reg_lambda': 1.5, 'classifier__reg_alpha': 0.01, 'classifier__n_estimators': 400}
Mejor F1-macro en CV: 0.6516

=== xgb_bow_features_rs_v8_2 ===
Hiperparámetros: {'classifier__subsample': 0.75, 'classifier__reg_lambda': 1.5, 'classifier__reg_alpha': 0.01, 'classifier__n_estimators': 400}

F1-macro:  0.6626
Precision: 0.6637
Recall:    0.6637
Accuracy:  0.7033

              precision    recall  f1-score   support

    negativa     0.7780    0.7517    0.7646      4080
      neutra     0.3997    0.4657    0.4302      2040
    positiva     0.8135    0.7738    0.7931      4080

    accuracy                         0.7033     10200
   macro avg     0.6637    0.6637    0.6626     10200
weighted avg     0.7165    0.7033    0.7091     10200

Matriz de confusión (filas=real, cols=predicho):
          negati

In [11]:
# Reentrenar en train completo
y_full = np.concatenate([y_train, y_val])
weights_full = compute_sample_weight("balanced", y_full)
best_pipe = rs.best_estimator_
best_pipe.fit(X_full_combined, y_full, sample_weight=weights_full)

,vectorizer,CountVectoriz...\w[\\w-]+\\b')
,classifier,"XGBClassifier...ree=None, ...)"
,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,1.0
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None


In [12]:
save_model(best_pipe, "xgb_bow_features_rs_v8_2")

Modelo guardado → models\xgb_bow_features_rs_v8_2.joblib


WindowsPath('models/xgb_bow_features_rs_v8_2.joblib')

In [13]:
make_submission(best_pipe,
                pd.DataFrame({"id": test["id"], "text": X_test}),
                "submission_xgb_bow_features_rs_v8_2.csv",
                raw_texts=test["text"].values);

Predicción guardada → submissions\submission_xgb_bow_features_rs_v8_2.csv  (8500 predicciones)
Distribución: clase 0: 38.7%, clase 1: 23.7%, clase 2: 37.6%
